
# Livrable 3 – Génération de légendes d’images (Image Captioning)
![Worklofw](../assets/Soutenance/workflow.png)
Ce livrable constitue la dernière étape du projet **Leyanda**. L'objectif est de développer un modèle de deep learning capable de générer automatiquement une légende textuelle en anglais pour une image, en s'appuyant sur le dataset **MS COCO**.

Le système repose sur un modèle **encodeur-décodeur** combinant un réseau de neurones convolutifs (CNN) et un réseau de neurones récurrents (RNN).

---

## Architecture du modèle

### 1. Encodeur : InceptionV3

L'image est encodée via un CNN InceptionV3 pré-entraîné sur ImageNet :
- Les couches de classification sont retirées.
- Une couche `GlobalAveragePooling2D` réduit les features.
- Une `Dense(relu)` convertit l'image en un vecteur de taille `embedding_dim` (ex: 256).

### 2. Décodeur : LSTM

Le vecteur image initialise les états `h` et `c` du LSTM.
- La séquence de mots (caption début) est vectorisée via une couche `Embedding`.
- Le LSTM prédit un mot à chaque pas de temps.
- `Dropout` et `Dense(softmax)` produisent la sortie.

### 3. Modèle combiné : image + texte → légende

```python
image_input = Input(shape=(180, 180, 3))
caption_input = Input(shape=(max_length,))

image_features = encoder(image_input)
embedding = Embedding(vocab_size, embedding_dim, mask_zero=True)(caption_input)

h_initial = Dense(units, activation='relu')(image_features)
c_initial = Dense(units, activation='relu')(image_features)

lstm_out = LSTM(units, return_sequences=True)(embedding, initial_state=[h_initial, c_initial])
dropout = Dropout(dropout_rate)(lstm_out)
output = Dense(vocab_size, activation='softmax')(dropout)

captioning_model = Model(inputs=[image_input, caption_input], outputs=output)
```

Ce modèle apprend à prédire la suite d'une légende à partir d'une image et d'une séquence déjà produite.

---

## Prétraitement

### Images
- Redimensionnées en `180x180` pixels
- Normalisation via `preprocess_input`
- Passent dans InceptionV3 pour extraction de features

### Texte
- Nettoyage des légendes (ponctuation, casses)
- Tokenisation avec un `Tokenizer` Keras
- Padding des séquences à `max_length`

---

## Entraînement et performances

Le modèle est entraîné avec la perte `SparseCategoricalCrossentropy` (avec masquage des zéros).

```python
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.legend()
```

> ⚠️ Le score BLEU était initialement calculé à chaque époque, mais cela ralentissait trop l'entraînement. Il est donc calculé uniquement lors de l'inférence.

---

## Inférence et résultats

Pour générer une légende :

```python
caption = generate_caption(image_path, model, tokenizer, max_length)
```

Le modèle prédit un mot à la fois jusqu'au token `<end>` ou `max_length`.

---